# 8 — Open-set energy gate, E5-lite (Phase 5, RQ4)

OpenSpec change `stream-statistics-adaptation`, design D9. Leave-one-class-out (LOCO) source models on centred
CLEAN features (seed 0, 5 folds, Phase 1 protocol, 6 outputs). Test streams contain the held-out class's recordings
as unknowns. Runs use the frozen `phase5/stream_hparams.json` (source and triggered), gate on and off.

Stream contents, fixed before running:
- CLEAN: the LOCO model's known test-fold recordings plus every CLEAN recording of the held-out class;
- BOTH: every seen-mode BOTH recording of the 6 known classes plus every seen-mode BOTH recording of the held-out
  class (no BOTH recording is in any training set). Recordings are chained in an order fixed by
  `default_rng([0, fold, 100 + c, condition])`; centring is the Phase 4 stream estimator over the whole stream.

In [1]:
import os
import json
import itertools
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from sklearn.metrics import f1_score, roc_auc_score
from tqdm import tqdm

from rfml_uav.adapt.stream import energy, run_stream
from rfml_uav.data.normalise import session_normalise, stream_normalise
from rfml_uav.drone_detect.dataset import DRONES
from rfml_uav.eval.splits import make_splits, undersample
from rfml_uav.nn.models import get_model
from rfml_uav.nn.train import train_best_val

load_dotenv()
DATA = os.getenv("DATA_PATH")
P1, P5 = os.path.join(DATA, "results/paper3/phase1"), os.path.join(DATA, "results/paper3/phase5")
CKPT = os.path.join(DATA, "checkpoints/paper3/loco")
os.makedirs(CKPT, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False

CLASSES = list(DRONES)
SEED, K_FOLDS = 0, 5
EPOCHS, BATCH_SIZE, LEARNING_RATE = 50, 256, 0.01  # Phase 1
BETA, K_WARM = 0.01, 50
CONDITIONS = ["CLEAN", "BOTH"]
with open(os.path.join(P5, "stream_hparams.json")) as fh:
    HPS = json.load(fh)["methods"]

In [2]:
F = dict(np.load(os.path.join(P1, "features.npz")))
F["y"] = np.array([CLASSES.index(d) for d in F["drone"]])
clean, active = F["subset"] == "CLEAN", F["active"]
seen = {(d, m) for d, m in zip(F["drone"][clean], F["mode"][clean])}
F["seen"] = np.array([(d, m) in seen for d, m in zip(F["drone"], F["mode"])])
REC = {r: rows[np.argsort(F["start"][rows], kind="stable")]
       for r, rows in pd.Series(np.arange(len(F["y"]))).groupby(F["rec_id"]).apply(np.array).items()}
XS = session_normalise(F["X"], F["rec_id"], ~active)
XS_t = torch.tensor(XS, device=device).unsqueeze(1)


def known_labels(c):
    """Labels in the 6-class space of the model without class c; c -> -1."""
    y = F["y"] - (F["y"] > c)
    y[F["y"] == c] = -1
    return y


def loco_split(c, fold):
    idx = undersample(np.flatnonzero(clean & active & (F["y"] != c)), F["y"], SEED)
    tr, va, te = make_splits(F["y"][idx], F["rec_id"][idx], "recording", n_folds=K_FOLDS, seed=SEED)[fold]
    return idx[tr], idx[va], idx[te]


def load_model(c, fold):
    m = get_model("CNN", input_dim=256, n_classes=len(CLASSES) - 1, linear_tail=True)
    m.load_state_dict(torch.load(os.path.join(CKPT, f"loco_c{c}_s{SEED}_f{fold}.pt"), map_location=device))
    return m.to(device).eval()

## LOCO models (35)

In [3]:
for c, fold in tqdm(list(itertools.product(range(len(CLASSES)), range(K_FOLDS)))):
    tr, va, te = loco_split(c, fold)
    assert not (F["y"][np.r_[tr, va, te]] == c).any()
    path = os.path.join(CKPT, f"loco_c{c}_s{SEED}_f{fold}.pt")
    if os.path.exists(path):
        continue
    y_t = torch.tensor(np.maximum(known_labels(c), 0), device=device)  # -1 rows are never indexed by tr / va
    torch.manual_seed(SEED)
    model = get_model("CNN", input_dim=256, n_classes=len(CLASSES) - 1, linear_tail=True).to(device)
    model, val_f1, epoch = train_best_val(model, XS_t, y_t, tr, va, epochs=EPOCHS, lr=LEARNING_RATE,
                                          batch_size=BATCH_SIZE, seed=SEED)
    torch.save(model.state_dict(), path)
assert len([f for f in os.listdir(CKPT) if f.endswith(".pt")]) == len(CLASSES) * K_FOLDS
print("35 LOCO checkpoints; no training or validation row belongs to the held-out class")

  0%|          | 0/35 [00:00<?, ?it/s]

  3%|▎         | 1/35 [00:10<05:43, 10.10s/it]

  6%|▌         | 2/35 [00:19<05:25,  9.87s/it]

  9%|▊         | 3/35 [00:29<05:11,  9.73s/it]

 11%|█▏        | 4/35 [00:38<04:57,  9.59s/it]

...

 94%|█████████▍| 33/35 [05:20<00:19,  9.83s/it]

 97%|█████████▋| 34/35 [05:29<00:09,  9.82s/it]

100%|██████████| 35/35 [05:39<00:00,  9.83s/it]

100%|██████████| 35/35 [05:39<00:00,  9.71s/it]

35 LOCO checkpoints; no training or validation row belongs to the held-out class


## Open-set streams

In [4]:
def build_stream(c, fold, condition):
    _, _, te = loco_split(c, fold)
    if condition == "CLEAN":
        known = sorted(set(F["rec_id"][te]))
    else:
        known = sorted(set(F["rec_id"][(F["subset"] == "BOTH") & F["seen"] & (F["y"] != c)]))
    unknown = sorted(set(F["rec_id"][(F["subset"] == condition) & F["seen"] & (F["y"] == c)]))
    recs = np.array(known + unknown)
    recs = recs[np.random.default_rng([SEED, fold, 100 + c, CONDITIONS.index(condition)]).permutation(len(recs))]
    rows = np.concatenate([REC[r] for r in recs])
    Xn, valid = stream_normalise(F["X"][rows], ~active[rows], beta=BETA, k=K_WARM)
    act = active[rows] & valid
    return {"X": Xn, "active": act, "y": known_labels(c)[rows][act], "rows": rows[act]}


def calib_energy(model, c, fold):
    _, va, _ = loco_split(c, fold)
    with torch.no_grad():
        return torch.cat([energy(model(XS_t[va[i:i + 4096]])) for i in range(0, len(va), 4096)]).cpu().numpy()


res_path = os.path.join(P5, "e5_lite_runs.csv")
if not os.path.exists(res_path):
    out = []
    for c, fold in tqdm(list(itertools.product(range(len(CLASSES)), range(K_FOLDS)))):
        model = load_model(c, fold)
        calib = calib_energy(model, c, fold)
        tr, va, _ = loco_split(c, fold)
        for condition in CONDITIONS:
            s = build_stream(c, fold, condition)
            assert not set(F["rec_id"][s["rows"]]) & set(F["rec_id"][np.r_[tr, va]])
            unk = s["y"] < 0
            assert unk.any() and (~unk).any()
            X = torch.as_tensor(s["X"], device=device).unsqueeze(1)
            for method, gate in itertools.product(["source", "triggered"], [True, False]):
                o = run_stream(model, X, s["active"], method, {**HPS[method], "gate": gate}, calib, seed=fold)
                k = ~unk & (o["known"] if gate else True)
                out.append({"held_out": CLASSES[c], "fold": fold, "condition": condition, "method": method,
                            "gate": gate, "unknown_share": unk.mean(),
                            "energy_auroc": roc_auc_score(unk, o["energy"]),
                            "unknown_false_accept": o["known"][unk].mean() if gate else 1.0,
                            "known_rejection": 1 - o["known"][~unk].mean() if gate else 0.0,
                            "known_vote_f1": f1_score(s["y"][k], o["vote"][k], average="macro",
                                                      labels=range(len(CLASSES) - 1)),
                            "events": o["cost"]["events"]})
    pd.DataFrame(out).to_csv(res_path, index=False)
runs = pd.read_csv(res_path)

  0%|          | 0/35 [00:00<?, ?it/s]

  3%|▎         | 1/35 [00:02<01:08,  2.01s/it]

  6%|▌         | 2/35 [00:03<01:04,  1.96s/it]

  9%|▊         | 3/35 [00:05<01:02,  1.94s/it]

 11%|█▏        | 4/35 [00:07<00:59,  1.94s/it]

...

 91%|█████████▏| 32/35 [01:02<00:05,  1.93s/it]

 94%|█████████▍| 33/35 [01:03<00:03,  1.92s/it]

 97%|█████████▋| 34/35 [01:05<00:01,  1.92s/it]

100%|██████████| 35/35 [01:07<00:00,  1.91s/it]

100%|██████████| 35/35 [01:07<00:00,  1.94s/it]

In [5]:
wide = runs.pivot_table(index=["held_out", "condition", "fold"], columns=["method", "gate"],
                        values=["known_vote_f1", "energy_auroc", "unknown_false_accept", "known_rejection", "events"])
e5 = pd.DataFrame({
    "energy_auroc": wide[("energy_auroc", "source", True)],
    "unknown_false_accept": wide[("unknown_false_accept", "source", True)],
    "known_rejection": wide[("known_rejection", "source", True)],
    "known_vote_f1_gate_on": wide[("known_vote_f1", "source", True)],
    "known_vote_f1_gate_off": wide[("known_vote_f1", "source", False)],
    "triggered_unknown_false_accept": wide[("unknown_false_accept", "triggered", True)],
    "triggered_known_vote_f1_gate_on": wide[("known_vote_f1", "triggered", True)],
    "triggered_known_vote_f1_gate_off": wide[("known_vote_f1", "triggered", False)],
    "triggered_events_gate_on": wide[("events", "triggered", True)],
}).groupby(level=["held_out", "condition"], sort=False).mean()
e5 = e5.reindex([(c, cond) for c in CLASSES for cond in CONDITIONS])
mean = e5.groupby(level="condition").mean()
mean.index = pd.MultiIndex.from_tuples([("mean", c) for c in mean.index], names=e5.index.names)
e5 = pd.concat([e5, mean])
assert len(e5) == len(CLASSES) * len(CONDITIONS) + len(CONDITIONS)
e5.to_csv(os.path.join(P5, "e5_lite.csv"))
e5.round(3)

energy_auroc  unknown_false_accept  known_rejection  \
held_out condition                                                        
AIR      CLEAN             0.595                 0.822            0.114   
         BOTH              0.450                 0.854            0.257   
DIS      CLEAN             0.643                 0.801            0.092   
         BOTH              0.577                 0.741            0.166   
INS      CLEAN             0.715                 0.788            0.100   
         BOTH              0.684                 0.713            0.172   
MIN      CLEAN             0.524                 0.920            0.113   
         BOTH              0.401                 0.887            0.201   
MP1      CLEAN             0.746                 0.689            0.118   
         BOTH              0.652                 0.724            0.171   
MP2      CLEAN             0.643                 0.873            0.119   
         BOTH              0.663                 0.675            0.194   
PHA      CLEAN             0.582                 0.939            0.145   
         BOTH              0.405                 0.931            0.265   
mean     BOTH              0.547                 0.789            0.204   
         CLEAN             0.636                 0.833            0.114   

                    known_vote_f1_gate_on  known_vote_f1_gate_off  \
held_out condition                                                  
AIR      CLEAN                      0.880                   0.855   
         BOTH                       0.694                   0.654   
DIS      CLEAN                      0.845                   0.825   
         BOTH                       0.736                   0.714   
INS      CLEAN                      0.852                   0.832   
         BOTH                       0.739                   0.721   
MIN      CLEAN                      0.827                   0.804   
         BOTH                       0.638                   0.610   
MP1      CLEAN                      0.887                   0.861   
         BOTH                       0.762                   0.734   
MP2      CLEAN                      0.900                   0.869   
         BOTH                       0.772                   0.748   
PHA      CLEAN                      0.868                   0.834   
         BOTH                       0.760                   0.719   
mean     BOTH                       0.729                   0.700   
         CLEAN                      0.866                   0.840   

                    triggered_unknown_false_accept  \
held_out condition                                   
AIR      CLEAN                               0.842   
         BOTH                                0.854   
DIS      CLEAN                               0.801   
         BOTH                                0.741   
INS      CLEAN                               0.810   
         BOTH                                0.713   
MIN      CLEAN                               0.920   
         BOTH                                0.887   
MP1      CLEAN                               0.689   
         BOTH                                0.724   
MP2      CLEAN                               0.873   
         BOTH                                0.675   
PHA      CLEAN                               0.916   
         BOTH                                0.931   
mean     BOTH                                0.789   
         CLEAN                               0.836   

                    triggered_known_vote_f1_gate_on  \
held_out condition                                    
AIR      CLEAN                                0.811   
         BOTH                                 0.694   
DIS      CLEAN                                0.845   
         BOTH                                 0.736   
INS      CLEAN                                0.805   
         BOTH                                 0.739   
MIN      CLEA